# Mini-Project 2: SHAP Deep Dive — Explaining a Loan Default Model

## Goal
Build a loan default classifier and deeply interpret it using SHAP (SHapley Additive exPlanations).

## What You'll Practice
- **Global feature importance** — which features drive defaults overall?
- **Local explanations** — why was *this* customer flagged?
- **Business-ready narratives** — translating model outputs into plain English

## Estimated Time: 2–3 hours

---

In [ ]:
import sys
sys.path.insert(0, '../../..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.preprocessing import StandardScaler
import shap
shap.initjs()

from shared.viz_utils import setup_style, save_fig
setup_style()

## Generate Synthetic Loan Dataset

We create a realistic loan dataset with **known relationships** between features and default.
This lets us verify that SHAP recovers the true drivers.

In [ ]:
np.random.seed(42)
n = 2000

data = pd.DataFrame({
    'annual_income': np.random.lognormal(mean=10.8, sigma=0.5, size=n).round(0),
    'loan_amount': np.random.lognormal(mean=9.5, sigma=0.6, size=n).round(0),
    'interest_rate': np.random.uniform(5, 25, n).round(2),
    'employment_length': np.random.exponential(5, n).clip(0, 30).round(1),
    'credit_score': np.random.normal(680, 60, n).clip(300, 850).astype(int),
    'dti_ratio': np.random.uniform(0.05, 0.65, n).round(3),
    'num_open_accounts': np.random.poisson(8, n),
    'delinquencies_2yr': np.random.poisson(0.5, n),
    'home_ownership': np.random.choice(['OWN', 'MORTGAGE', 'RENT'], n, p=[0.15, 0.45, 0.40]),
    'loan_purpose': np.random.choice(['debt_consolidation', 'home_improvement', 'major_purchase', 'other'], n, p=[0.45, 0.20, 0.15, 0.20]),
})

# Create default with known relationships
logit = (
    -2.0
    - 0.00003 * data['annual_income']
    + 0.00004 * data['loan_amount']
    + 0.08 * data['interest_rate']
    - 0.05 * data['employment_length']
    - 0.01 * data['credit_score']
    + 2.0 * data['dti_ratio']
    + 0.3 * data['delinquencies_2yr']
    + 0.3 * (data['home_ownership'] == 'RENT')
    + np.random.normal(0, 0.5, n)
)
data['defaulted'] = (np.random.random(n) < 1 / (1 + np.exp(-logit))).astype(int)
print(f"Default rate: {data['defaulted'].mean():.1%}")
data.head()

---

## Part 1: Quick Model

Train a model quickly — the focus of this project is **interpretation, not model tuning**.

In [ ]:
# Encode categoricals
df = pd.get_dummies(data, columns=['home_ownership', 'loan_purpose'], drop_first=True)

X = df.drop('defaulted', axis=1)
y = df['defaulted']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model = GradientBoostingClassifier(n_estimators=200, max_depth=4, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))
print(f"ROC-AUC: {roc_auc_score(y_test, model.predict_proba(X_test)[:, 1]):.4f}")

---

## Part 2: Global Interpretability

Use SHAP to understand **which features drive defaults overall**.

SHAP values decompose each prediction into per-feature contributions. Summing across all samples gives us a global picture.

In [ ]:
# TODO: Create a TreeExplainer
# explainer = shap.TreeExplainer(model)

# TODO: Compute SHAP values for the test set
# shap_values = explainer.shap_values(X_test)
# Note: For binary classification, shap_values may be a list [class_0, class_1]
# Use shap_values[1] for the positive class (default)

In [ ]:
# TODO: Create a SHAP beeswarm summary plot
# shap.summary_plot(shap_values[1], X_test, show=False)
# plt.title('SHAP Summary — Loan Default')
# plt.tight_layout()
# plt.show()

# TODO: Create a SHAP bar plot (mean |SHAP value|)
# shap.summary_plot(shap_values[1], X_test, plot_type="bar", show=False)
# plt.title('Mean |SHAP Value| by Feature')
# plt.tight_layout()
# plt.show()

**TODO: Answer these questions based on your summary plots:**

1. What are the top 3 most important features?
2. For the #1 feature, does a HIGH value increase or decrease default risk?
3. Are there any surprising features in the top 5?

---

## Part 3: Dependence Plots

How does each feature **individually** affect the prediction?

Dependence plots show the SHAP value for a single feature (y-axis) against the feature's actual value (x-axis). SHAP automatically colors by the strongest interacting feature.

In [ ]:
# TODO: Create SHAP dependence plots for the top 3 features
# shap.dependence_plot('feature_name', shap_values[1], X_test, show=False)
# plt.tight_layout()
# plt.show()

# TODO: Look at the automatic interaction coloring
# What feature does SHAP choose to color by? What does this tell you?

---

## Part 4: Local Explanations

Why did the model predict default for **specific customers**?

This is the most powerful part of SHAP — you can explain individual predictions to stakeholders.

In [ ]:
# TODO: Find one clear default prediction and one clear non-default
# probs = model.predict_proba(X_test)[:, 1]
# high_risk_idx = probs.argmax()   # most likely to default
# low_risk_idx = probs.argmin()    # least likely to default
#
# print(f"High risk customer: P(default) = {probs[high_risk_idx]:.4f}")
# print(f"Low risk customer: P(default) = {probs[low_risk_idx]:.4f}")

In [ ]:
# TODO: Waterfall plot for the high-risk customer
# Use shap.Explanation object:
# explanation = shap.Explanation(
#     values=shap_values[1][high_risk_idx],
#     base_values=explainer.expected_value[1],
#     data=X_test.iloc[high_risk_idx],
#     feature_names=X_test.columns.tolist()
# )
# shap.waterfall_plot(explanation, show=False)
# plt.title('High Risk Customer')
# plt.tight_layout()
# plt.show()

# TODO: Repeat for low-risk customer

**TODO: Write a plain-English explanation for a loan officer:**

**High-risk customer:**
"This customer was flagged as high default risk because..."

**Low-risk customer:**
"This customer was classified as low risk because..."

---

## Part 5: Cohort Analysis

Do different customer segments have **different risk drivers**?

Comparing SHAP values across cohorts can reveal that the model relies on different features for different groups — important for fairness and targeted interventions.

In [ ]:
# TODO: Split test data by home ownership (RENT vs not RENT)
# Compute mean |SHAP values| for each group
# Compare: which features matter more for renters vs homeowners?

# Hint:
# rent_mask = X_test['home_ownership_RENT'] == 1
# rent_shap = np.abs(shap_values[1][rent_mask]).mean(axis=0)
# own_shap = np.abs(shap_values[1][~rent_mask]).mean(axis=0)
#
# comparison = pd.DataFrame({'Renters': rent_shap, 'Owners': own_shap}, index=X_test.columns)
# comparison.sort_values('Renters', ascending=True).plot(kind='barh', figsize=(10, 6))
# plt.title('Mean |SHAP Value| — Renters vs Owners')
# plt.xlabel('Mean |SHAP Value|')
# plt.tight_layout()
# plt.show()

---

## Part 6: SHAP Interaction Values (Stretch Goal)

SHAP interaction values decompose each SHAP value further into main effects + pairwise interactions. This is computationally expensive but reveals which feature *pairs* matter most.

In [ ]:
# STRETCH GOAL: Compute SHAP interaction values (slow!)
# interaction_values = explainer.shap_interaction_values(X_test[:100])
# shap.summary_plot(interaction_values[1], X_test[:100], show=False)
# plt.tight_layout()
# plt.show()

---

## Key Findings

TODO: Summarize your SHAP analysis:

1. **Most important features:** 
2. **Key insight from dependence plots:** 
3. **Difference between customer segments:** 
4. **Actionable recommendation for the business:** 

## What I Learned
- 